# Lesson 13: What can generative AI do? — make an AI picture book 📚
**Learning goals**
1. See one model handle many tasks: writing, translating, summarising, coding
2. Chain **text + images + voice** AIs into a "production line"
3. Think: which steps should AI do, and which must a human do?

**⏱️ Timing:** many tasks 15' | picture-book pipeline 30' | group design 15'

> ⚙️ **Before you start:** go to **Runtime → Change runtime type → T4 GPU**, then run the cells from top to bottom.

## 🏭 An analogy first
Making a picture book used to take: an author writes the story → a translator → an illustrator draws → a narrator records the audio.
Today we'll build a **production line** where each station is an AI — and **you are the editor-in-chief**.

> Real-world uses: writing help, translation, coding assistants (programmers use them every day), design, voice-over, medical research (AI designing new drug molecules, predicting protein structures)…

In [ ]:
# ⏳ The first run downloads the model (about 1–3 minutes). Be patient!
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL = "Qwen/Qwen2.5-1.5B-Instruct" if torch.cuda.is_available() else "Qwen/Qwen2.5-0.5B-Instruct"
tok = AutoTokenizer.from_pretrained(MODEL)
llm = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype="auto", device_map="auto")

def ask(prompt, system="You are a friendly assistant. Answer in simple English.",
        max_new_tokens=300, temperature=0.7):
    """Send a prompt to the AI and return its answer."""
    messages = [{"role": "system", "content": system},
                {"role": "user", "content": prompt}]
    text = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tok(text, return_tensors="pt").to(llm.device)
    out = llm.generate(**inputs, max_new_tokens=max_new_tokens,
                       do_sample=temperature > 0, temperature=temperature if temperature > 0 else None)
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

print("✅ Model loaded:", MODEL)

## Part 1: One model, many skills 🛠️

In [ ]:
tasks = {
    "✍️ Writing": "Write a 50-word description of autumn.",
    "🌏 Translation": "Translate into Spanish: Knowledge is power, but imagination is more important than knowledge.",
    "📝 Summary": "Summarise in one sentence: Photosynthesis is the process plants use to turn sunlight, water and carbon "
                 "dioxide into glucose and oxygen. It happens in the chloroplasts and is the energy source for almost all life on Earth.",
    "💻 Coding": "Write Python code that prints all even numbers from 1 to 20. Only give the code.",
}
for name, t in tasks.items():
    print(f"{name}\n{ask(t)}\n")

### 🧩 Exercise 1: Does the AI's code actually work?
Copy the AI's code into the cell below and run it. Does it run? Is the result correct?
**Important habit:** always **run and check** code that AI writes for you.

In [ ]:
# paste the AI's code here

## Part 2: The AI picture-book pipeline 📖
**Station 1: the writer** — ask the AI for a 4-page story, with a one-sentence scene description for each page (for the illustrator).

In [ ]:
topic = "a little firefly who is afraid of the dark"   # 👉 change to your group's topic

story_prompt = f"""Write a 4-page picture-book story for 8-year-olds about: {topic}.
Use exactly this format for all 4 pages and write nothing else:
Page 1: (one or two sentences of story)
Scene: (one sentence describing the picture, for an AI image generator)
Page 2: ...
Scene: ..."""
story = ask(story_prompt, max_new_tokens=500)
print(story)

In [ ]:
# Split the AI's output into [(story, scene), ...]
import re
pages = re.findall(r"Page \d+:\s*(.+?)\s*Scene:\s*(.+)", story)
for i, (text, scene) in enumerate(pages, 1):
    print(i, text, "|", scene)
print(len(pages), "pages found (if it isn't 4, the AI ignored the format — rerun the cell above or fix it by hand)")

**Station 2: the illustrator** — use the diffusion model from Lesson 10 to draw the pictures.

In [ ]:
!pip -q install diffusers gTTS
import torch, matplotlib.pyplot as plt
from diffusers import AutoPipelineForText2Image
painter = AutoPipelineForText2Image.from_pretrained(
    "stabilityai/sd-turbo", torch_dtype=torch.float16, variant="fp16").to("cuda")

style = ", children's picture book illustration, soft colors"   # one consistent style
images = [painter(scene + style, num_inference_steps=1, guidance_scale=0.0).images[0] for _, scene in pages]

fig, axes = plt.subplots(1, len(images), figsize=(4 * len(images), 4), squeeze=False)
for ax, im in zip(axes.flat, images):
    ax.imshow(im); ax.axis("off")
plt.show()

**Station 3: the narrator** — read the story aloud.

In [ ]:
from gtts import gTTS
from IPython.display import Audio, display

full_text = " ".join(t for t, _ in pages)
gTTS(full_text, lang="en").save("story.mp3")
display(Audio("story.mp3"))

**Editor-in-chief (that's you!)** — lay out each page with its picture.

In [ ]:
for i, ((text, _), im) in enumerate(zip(pages, images), 1):
    plt.figure(figsize=(5, 5)); plt.imshow(im); plt.axis("off")
    plt.title(f"Page {i}"); plt.show()
    print(f"Page {i}: {text}\n")

### 🧩 Exercise 2: The editor's job
1. Does the story contradict itself anywhere? Do the pictures match the text? Does the main character look the same on every page?
2. **Fix at least one thing by hand**: rewrite a sentence, or change a scene description and redraw it.
3. Record: what did the AI do well? What had to be fixed by a human?

### 🚀 Challenge
- Make a second version of the story in another language and read it aloud with `gTTS(..., lang="es")` (or `"fr"`, `"zh-CN"`…)
- Fix the "main character looks different on every page" problem (hint: put the same character description at the start of every scene, and fix the seed)

## Part 3: Group design 💡
Use generative AI to solve a **real problem at your school**. Fill in your plan below:

In [ ]:
plan = {
    "The problem we're solving": "",
    "Who will use it": "",
    "What the AI does": "",
    "What humans do": "",
    "What could go wrong": "",
}
for k, v in plan.items(): print(f"{k}: {v}")

## 📌 Summary
- Generative AI can write, translate, draw, speak and code
- Several AIs can be chained into a **pipeline**, but a human needs to **check, fix and decide**
- AI is the tool; **you** are the editor-in-chief